# Accelerated "deconvolution" of projected images using CARE

This notebook demonstrates applying a CARE model for a 2D denoising task, assuming that training was already completed via [2_training.ipynb](2_training.ipynb), or that a pre-trained model is available.
 
**We provide a pre-trained model**, which has been trained of an heterogenous set of ISS data coming from 2 different microscopes (Leica and Zeiss in our lab), magnifications (20x and 40x) and tissues (fly embryos and ovaries, developing human spinal cord and mouse brains). 

The model is available at `https://github.com/Moldia/ISS2025/tree/main/ISS_CARE/models/spots`.

Our models have been benchmarked using chicken brain data (never seen in the training process), and show excellent performance and a great speed tradeoff Vs canonical deconvolution. 

**However, we strongly advise training your own model, especially if you don't belong to Mats Nilsson's lab. Your microscope and data might be significantly different from ours, and our models could potentially produce artifacts in your data.** 

More documentation about how CARE works, and about how to train your own models, is available at http://csbdeep.bioimagecomputing.com/doc/.

In our lab's workflow, CARE-mediate deconvolution should be applied just after the preprocessing step (without deconvolution; `deconvolution_method = None`). 

For this notebook to work correctly, you should have a specific folder for each one of the regions you want to process, `/R1/`, `/R2/`, etc. Each region folder should contain the following subfolders tree: `/preprocessing/CycleX/4_retiled/`. Under `/preprocessing/` you might still have other subfolders such as `/1_mipped/`, `/2_ome_tiffs/` and `/3_stitched/`, but these are irrelevant at this stage.

The important thing is that you have a `/preprocessing/CycleX/4_retiled/`, because the retiled images are the starting point of the CARE-mediate deconvolution process.




## We start by importing the necessary modules

In [9]:
from ISS_CARE.ISS_CARE_prediction import ISS_CARE_predict

## Denoising prediction with CARE

### Core parameters
`input_dir` (str): Path to parent directory containing region folders (`R1/`, `R2/`, …).  
`regions_to_process` (list[int] | None): 1-based region indices; `None` means all detected regions are processed.  
`output_dir_prefix` (str | None): Optional output root. If `None`, outputs are saved inside the input structure at `input_dir/R#/preprocessing/CycleX/4_retiled/CARE/`. If set, outputs are saved to `output_dir_prefix/R#/preprocessing/CycleX/4_retiled/CARE/`.  
`model_dir` (str | Path): Directory containing trained CARE models.  
`model_name` (str): Model subfolder name.  
`dapi_ch` (int, zero-based): DAPI channel index. Files matching `*_ch{dapi_ch}.tif` are copied unchanged; all other channels are denoised with CARE.

### Normalization behavior
`normalize_input` (bool | None, default=`None`): Controls whether percentile normalization is applied before prediction.  
- `None` → automatically check `training_metadata.json` in the model folder and use the training setting if available  
- `False` → use raw intensities  
- `True` → apply percentile normalization before prediction using `normalized = (x - pmin) / (pmax - pmin + eps)`

`normalization_pmin` (float | None, default=`None`): lower percentile. If `None`, try to load from training metadata; otherwise default to `1.0`.  
`normalization_pmax` (float | None, default=`None`): upper percentile. If `None`, try to load from training metadata; otherwise default to `99.8`.  
`normalization_eps` (float | None, default=`None`): numerical stability term. If `None`, try to load from training metadata; otherwise default to `1e-8`.

**Important:** if normalization is used, predictions are rescaled before saving using `restored = restored * 65535`.

### Training metadata auto-detection
The function looks for:  
`model_dir/model_name/training_metadata.json`

If found, it can automatically read:
- whether normalization was enabled during training
- `PMIN`
- `PMAX`
- `EPS`

If the file is missing or unreadable, the function safely falls back to raw prediction (`normalize_input=False`) unless you explicitly override it.

### Debugging
`debug_prints` (bool, default=`True`): Print intensity statistics.  
`debug_print_limit` (int, default=`3`): Number of images per cycle to print debug stats for.  
Debug output can include raw input stats, normalized input stats (if enabled), prediction stats, and final uint16 output stats.

### Output behavior
Outputs are always saved as `uint16` TIFFs. Existing valid outputs are never overwritten. DAPI channels are copied unchanged. CSV metadata is preserved. An XML provenance file is written only if new outputs are generated.

### Recommended usage
Best default:
`normalize_input = None`

This lets inference automatically follow the training settings stored in the model metadata.

If you want to force behavior manually:
- use `normalize_input = False` for raw-intensity inference
- use `normalize_input = True` only if the model was trained with normalization

In [10]:
input_dir = '/path/to/regions/'
model_dir = '/path/to/model/' # /home/<user>/ISS2025/ISS_CARE/models/
model_name = 'CARE__NON_DAPI_train_patches_Leica_20_40__2026-04-18_02-30__2026-04-21_11-27'

In [11]:
input_dir = '/home/sagah/moldia-archive/CARE_training/test_organoid_data_leica-40X/raw/'
model_dir = '/home/sagah/moldia-archive/CARE_training/care_models/'
model_name = 'CARE__DAPI_ONLY_train_patches_Leica_40_signal_biased__2026-04-22_14-47__2026-04-22_15-39'

In [12]:
ISS_CARE_predict(
    input_dir=input_dir,
    model_dir=model_dir,
    model_name=model_name,
    dapi_ch=4,
    regions_to_process=None,
    output_dir_prefix=None,
    normalize_input=None,
    normalization_pmin=None,
    normalization_pmax=None,
    normalization_eps=None,
    debug_prints=True,
    debug_print_limit=3,
    visualize_debug_predictions=True,
    overwrite=True,
)

[INFO] Processing directory: /home/sagah/moldia-archive/CARE_training/test_organoid_data_leica-40X/raw
[INFO] Model base directory: /home/sagah/moldia-archive/CARE_training/care_models
[INFO] Model name: CARE__DAPI_ONLY_train_patches_Leica_40_signal_biased__2026-04-22_14-47__2026-04-22_15-39
[INFO] Training metadata file: /home/sagah/moldia-archive/CARE_training/care_models/CARE__DAPI_ONLY_train_patches_Leica_40_signal_biased__2026-04-22_14-47__2026-04-22_15-39/training_metadata.json
[INFO] normalize_input (resolved): False
[INFO] Using raw input intensities for prediction.
[INFO] debug_prints: True
[INFO] debug_print_limit: 3
[INFO] visualize_debug_predictions: True
[INFO] overwrite: True
[INFO] Using default output location under each region directory
[INFO] Regions found on disk (1): ['R1']
[INFO] Regions selected (1): ['R1']
Loading network weights from 'weights_best.h5'.
[INFO] Loaded CARE model: CARE__DAPI_ONLY_train_patches_Leica_40_signal_biased__2026-04-22_14-47__2026-04-22_15

100%|████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.99it/s]


[DEBUG] R1/Cycle1/Cycle1_s0_ch0.tif
[DEBUG] raw_input: dtype=float32, shape=(6000, 6000), min=0, max=4833, mean=225.527, std=142.703, p1=0, p50=170, p99=822
[DEBUG] model_input_raw: dtype=float32, shape=(6000, 6000), min=0, max=4833, mean=225.527, std=142.703, p1=0, p50=170, p99=822
[DEBUG] prediction_before_uint16: dtype=float32, shape=(6000, 6000), min=-147.479, max=9723.9, mean=213.102, std=188.335, p1=76.7231, p50=168.139, p99=1104.15
[DEBUG] saved_uint16_output: dtype=uint16, shape=(6000, 6000), min=0, max=9723, mean=212.603, std=188.333, p1=76, p50=168, p99=1104
[DEBUG] scale_check (R1/Cycle1/Cycle1_s0_ch0.tif)
[DEBUG]   normalize_input=False
[DEBUG]   raw_input_max=4833
[DEBUG]   model_input_max=4833
[DEBUG]   prediction_float_max=9723.9
[DEBUG]   prediction_float_mean=213.102
[DEBUG]   saved_uint16_max=9723
[DEBUG]   raw_fraction_gt0=0.9887, model_input_fraction_gt0=0.9887, prediction_fraction_gt0=0.9998
[INFO] Debug visualization written: /home/sagah/moldia-archive/CARE_traini

100%|████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00,  4.02it/s]


[DEBUG] R1/Cycle1/Cycle1_s0_ch1.tif
[DEBUG] raw_input: dtype=float32, shape=(6000, 6000), min=0, max=5327, mean=220.754, std=188.377, p1=0, p50=140, p99=1013
[DEBUG] model_input_raw: dtype=float32, shape=(6000, 6000), min=0, max=5327, mean=220.754, std=188.377, p1=0, p50=140, p99=1013
[DEBUG] prediction_before_uint16: dtype=float32, shape=(6000, 6000), min=-177.069, max=9686.17, mean=216.292, std=238.737, p1=54.8385, p50=160.885, p99=1348.08
[DEBUG] saved_uint16_output: dtype=uint16, shape=(6000, 6000), min=0, max=9686, mean=215.792, std=238.736, p1=54, p50=160, p99=1348
[DEBUG] scale_check (R1/Cycle1/Cycle1_s0_ch1.tif)
[DEBUG]   normalize_input=False
[DEBUG]   raw_input_max=5327
[DEBUG]   model_input_max=5327
[DEBUG]   prediction_float_max=9686.17
[DEBUG]   prediction_float_mean=216.292
[DEBUG]   saved_uint16_max=9686
[DEBUG]   raw_fraction_gt0=0.9887, model_input_fraction_gt0=0.9887, prediction_fraction_gt0=0.9998
[INFO] Debug visualization written: /home/sagah/moldia-archive/CARE_tr

...

[DEBUG] R1/Cycle5/Cycle5_s0_ch1.tif
[DEBUG] raw_input: dtype=float32, shape=(6000, 6000), min=0, max=5765, mean=268.407, std=248.874, p1=0, p50=165, p99=1292
[DEBUG] model_input_raw: dtype=float32, shape=(6000, 6000), min=0, max=5765, mean=268.407, std=248.874, p1=0, p50=165, p99=1292
[DEBUG] prediction_before_uint16: dtype=float32, shape=(6000, 6000), min=-502.465, max=11313.9, mean=196.243, std=322.223, p1=28.424, p50=100.197, p99=1739.28
[DEBUG] saved_uint16_output: dtype=uint16, shape=(6000, 6000), min=0, max=11313, mean=195.907, std=322.1, p1=28, p50=100, p99=1739
[DEBUG] scale_check (R1/Cycle5/Cycle5_s0_ch1.tif)
[DEBUG]   normalize_input=False
[DEBUG]   raw_input_max=5765
[DEBUG]   model_input_max=5765
[DEBUG]   prediction_float_max=11313.9
[DEBUG]   prediction_float_mean=196.243
[DEBUG]   saved_uint16_max=11313
[DEBUG]   raw_fraction_gt0=0.9775, model_input_fraction_gt0=0.9775, prediction_fraction_gt0=0.9956
[INFO] Debug visualization written: /home/sagah/moldia-archive/CARE_tra

100%|████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00,  4.14it/s]


[DEBUG] R1/Cycle5/Cycle5_s0_ch2.tif
[DEBUG] raw_input: dtype=float32, shape=(6000, 6000), min=0, max=8752, mean=294.708, std=267.426, p1=0, p50=192, p99=1357
[DEBUG] model_input_raw: dtype=float32, shape=(6000, 6000), min=0, max=8752, mean=294.708, std=267.426, p1=0, p50=192, p99=1357
[DEBUG] prediction_before_uint16: dtype=float32, shape=(6000, 6000), min=-528.598, max=17771.1, mean=213.913, std=320.396, p1=55.7376, p50=115.481, p99=1720.71
[DEBUG] saved_uint16_output: dtype=uint16, shape=(6000, 6000), min=0, max=17771, mean=213.48, std=320.346, p1=55, p50=115, p99=1720
[DEBUG] scale_check (R1/Cycle5/Cycle5_s0_ch2.tif)
[DEBUG]   normalize_input=False
[DEBUG]   raw_input_max=8752
[DEBUG]   model_input_max=8752
[DEBUG]   prediction_float_max=17771.1
[DEBUG]   prediction_float_mean=213.913
[DEBUG]   saved_uint16_max=17771
[DEBUG]   raw_fraction_gt0=0.9775, model_input_fraction_gt0=0.9775, prediction_fraction_gt0=0.9983
[INFO] Debug visualization written: /home/sagah/moldia-archive/CARE_t

100%|████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00,  4.05it/s]


[INFO] R1/Cycle5: predicted=20, copied_dapi=4, skipped_existing=0, copied_csv=1
[INFO] XML written: /home/sagah/moldia-archive/CARE_training/test_organoid_data_leica-40X/raw/R1/preprocessing/Cycle5/4_retiled/CARE/CARE_run_2026-04-22T14-32-08Z.xml
